In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
# --- Phase 1: Load data ---
expr = pd.read_csv('../data/raw/HiSeqV2', sep='\t', index_col=0)
mut = pd.read_csv('../data/raw/BRCA_mc3_gene_level.txt', sep='\t', index_col=0)
clin = pd.read_csv('../data/raw/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix', sep='\t', index_col=0)

In [3]:
# --- Phase 2: Find common patients, align ---
expr_samples = set(expr.columns)
mut_samples = set(mut.columns)
labeled_patients = set(clin[clin['PAM50Call_RNAseq'].notna()].index)
common = expr_samples & mut_samples & labeled_patients
common_patients = sorted(common)

expr_subset = expr[common_patients].T
mut_subset = mut[common_patients].T
labels = clin.loc[common_patients, 'PAM50Call_RNAseq']

In [4]:
# --- Phase 3: Feature filtering ---
gene_variances = expr_subset.var(axis=0)
top_genes = gene_variances.sort_values(ascending=False).head(2000).index

# Add back important PAM50 marker genes missing from top 2000
missing_markers = ['ERBB2', 'MKI67']
final_genes = list(top_genes) + missing_markers
expr_filtered = expr_subset[final_genes]
print("Expression filtered (with markers):", expr_filtered.shape)

min_patients = int(0.01 * len(mut_subset))
mut_filtered = mut_subset.loc[:, (mut_subset.sum(axis=0) >= min_patients)]
print("Mutation filtered:", mut_filtered.shape)

# Labels
le = LabelEncoder()
y = le.fit_transform(labels)

# Scaling
scaler = StandardScaler()
expr_scaled = scaler.fit_transform(expr_filtered)

# Train/val/test split
X_expr_train, X_expr_temp, X_mut_train, X_mut_temp, y_train, y_temp = train_test_split(
    expr_scaled, mut_filtered.values, y, test_size=0.3, stratify=y, random_state=42
)
X_expr_val, X_expr_test, X_mut_val, X_mut_test, y_val, y_test = train_test_split(
    X_expr_temp, X_mut_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_expr_train.shape, X_mut_train.shape, y_train.shape)
print("Val:", X_expr_val.shape, X_mut_val.shape, y_val.shape)
print("Test:", X_expr_test.shape, X_mut_test.shape, y_test.shape)

Expression filtered (with markers): (593, 2002)
Mutation filtered: (593, 2714)
Train: (415, 2002) (415, 2714) (415,)
Val: (89, 2002) (89, 2714) (89,)
Test: (89, 2002) (89, 2714) (89,)


In [5]:
# --- Phase 4: XGBoost baseline ---
X_train = np.hstack([X_expr_train, X_mut_train])
X_val = np.hstack([X_expr_val, X_mut_val])
X_test = np.hstack([X_expr_test, X_mut_test])

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(X_train, y_train, sample_weight=sample_weights, eval_set=[(X_val, y_val)], verbose=False)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       Basal       0.94      0.94      0.94        16
        Her2       1.00      0.71      0.83         7
        LumA       0.80      0.98      0.88        45
        LumB       0.92      0.61      0.73        18
      Normal       0.00      0.00      0.00         3

    accuracy                           0.84        89
   macro avg       0.73      0.65      0.68        89
weighted avg       0.84      0.84      0.83        89

Confusion matrix:
[[15  0  1  0  0]
 [ 0  5  1  1  0]
 [ 0  0 44  0  1]
 [ 0  0  7 11  0]
 [ 1  0  2  0  0]]


In [6]:
pam50_markers = ['ESR1', 'ERBB2', 'MKI67', 'PGR', 'EGFR', 'KRT5', 'KRT14']
for gene in pam50_markers:
    in_top2000 = gene in top_genes
    in_full = gene in expr_subset.columns
    print(f"{gene}: in_full_data={in_full}, in_top2000={in_top2000}")

ESR1: in_full_data=True, in_top2000=True
ERBB2: in_full_data=True, in_top2000=False
MKI67: in_full_data=True, in_top2000=False
PGR: in_full_data=True, in_top2000=True
EGFR: in_full_data=True, in_top2000=True
KRT5: in_full_data=True, in_top2000=True
KRT14: in_full_data=True, in_top2000=True


In [7]:
import pickle, os, numpy as np
os.makedirs('../results/models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

with open('../results/models/xgboost_model.pkl', 'wb') as f:
    pickle.dump(model, f)  # use whatever variable name your XGBoost model is stored in

np.savez('../data/processed/test_data.npz',
         X_expr_test=X_expr_test, X_mut_test=X_mut_test, y_test=y_test, X_test_concat=X_test)

with open('../data/processed/metadata.pkl', 'wb') as f:
    pickle.dump({'expr_features': list(expr_filtered.columns),
                  'mut_features': list(mut_filtered.columns),
                  'label_classes': le.classes_,
                  'cm_xgboost': confusion_matrix(y_test, model.predict(X_test))}, f)

print("Saved:", os.listdir('../results/models'), os.listdir('../data/processed'))

Saved: ['autoencoder_config.pt', 'autoencoder_state.pt', 'xgboost_model.pkl'] ['expression_clean.csv', 'labels.csv', 'metadata.pkl', 'mutation_clean.csv', 'test_data.npz']


In [8]:
import pickle, os
from sklearn.metrics import confusion_matrix

with open('../results/models/xgboost_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../data/processed/metadata.pkl', 'rb') as f:
    meta = pickle.load(f)

meta['cm_xgboost'] = confusion_matrix(y_test, model.predict(X_test))

with open('../data/processed/metadata.pkl', 'wb') as f:
    pickle.dump(meta, f)

print("Done")

Done
